# Applied Machine Learning, Notebook 5
## Credit risk: predicting a yes or a no with logistic regression

**Course:** *Applied Machine Learning* · **Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### A different kind of problem

Until now we always predicted a **number**: sales, return, volatility, price. Today the question is a
**yes or no**: will this customer repay the loan or not? This is called a **classification** problem,
and it is probably the most widespread use of machine learning in finance: every bank has a model that
does exactly this before approving a credit card or a loan.

We are going to use a single dataset, real and public: **German Credit** (Hofmann, 1994), 1,000 consumer
loans from a German bank, of which 300 ended up in default. And we are going to use a single algorithm,
**logistic regression**, from start to finish. That is on purpose: what you learn today about how to evaluate
a classifier, how to choose the threshold, what to do with imbalanced classes and how much to trust a
metric applies equally to any other model, and it is best learned without having to learn another
algorithm at the same time.

This notebook is the Python counterpart of three spreadsheets from the course: `logistica_01_german_credit.xlsx`
(the base model), `logistica_02_regularizacion.xlsx` and `logistica_03_desbalanceo.xlsx`. It uses exactly
the same data, the same preparation and the same split, so the numbers have to match cell by cell; if you
run Solver in the Excel file, you should arrive at the same coefficients as here.

### What you will learn today

1. Why a linear regression does not work for a yes or no, and how the **logistic function** fixes it.
2. How a model that predicts probabilities measures its errors: **cross-entropy**.
3. How a classifier is evaluated: accuracy, **confusion matrix**, **recall** (effectiveness) and
   **precision** (efficiency), and the trade-off between the two.
4. Why the 50% **threshold** is a decision and not a law: **ROC** curve, **AUC** and the cost-optimal threshold.
5. What **regularization** is in a logistic regression and what it actually does on this dataset.
6. What **class imbalance** causes and the four ways to correct it.
7. How much a metric moves if the sample changes: **cross-validation** and **confidence intervals**.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (roc_auc_score, roc_curve, log_loss, confusion_matrix, classification_report,
                             accuracy_score, recall_score, precision_score, f1_score)
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, train_test_split
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)


## 1. The data and its preparation

German Credit has 20 columns per customer, as the bank recorded them at the time of the application:
how much was requested, over how many months, how much was in the checking account and in savings, credit history, age,
housing, purpose of the loan and others. The last column, `class`, says whether the customer turned out to be a good
payer (`good`, 700 cases) or a bad payer (`bad`, 300 cases).

The next cell downloads the dataset from OpenML. If you have the course file (`german_credit.csv`),
paste its link into `URL_GERMAN` and that one is used instead of downloading.


In [ ]:

URL_GERMAN = ""    # <-- paste here the link to the course CSV (german_credit.csv) if you have it

def normalizar_link(url):
    '''Converts a Dropbox or Google Drive link into a direct-download link.'''
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if URL_GERMAN:
    german = pd.read_csv(normalizar_link(URL_GERMAN))
else:
    german = fetch_openml("credit-g", version=1, as_frame=True).frame
    for columna in german.select_dtypes("category").columns:
        german[columna] = german[columna].astype(str)
german = german.reset_index(drop=True)
german.insert(0, "id", np.arange(1, len(german) + 1))
print("Customers:", len(german), "  Columns:", german.shape[1] - 1)
print("Proportion of bad payers:", round((german["class"] == "bad").mean(), 3))
german[["id", "duration", "credit_amount", "age", "checking_status", "credit_history", "savings_status", "class"]].head(8)


### Step 1: transform and clean

A linear model can only multiply numbers by weights, so the columns with words are converted
into 0/1 columns, one per category, leaving one category out as the **base** (if all the dummies in a
group are 0, the customer is in the base category). The target variable becomes $y = 1$ if the customer was a bad
payer: that is the event we want to detect, so we call it the **positive** class.

Of the 20 columns we keep 9, chosen because they are the ones a credit analyst would look at first, and they
become 4 numeric columns and 14 0/1 columns. It is the same selection as in the Excel file; at the end of the notebook there is an
exercise with all 20.


In [ ]:

T = pd.DataFrame({"id": german["id"], "y": (german["class"] == "bad").astype(int)})
T["duracion"] = german["duration"].astype(float)                     # months
T["monto"] = german["credit_amount"].astype(float)                   # Deutsche Marks (DM)
T["edad"] = german["age"].astype(float)
T["cuota_pct_ingreso"] = german["installment_commitment"].astype(float)   # bracket 1 to 4 of the installment as % of income

cs = german["checking_status"]                                       # base: has no checking account
T["cta_negativa"] = (cs == "<0").astype(int)
T["cta_0_200"] = (cs == "0<=X<200").astype(int)
T["cta_sobre_200"] = (cs == ">=200").astype(int)
ch = german["credit_history"]                                        # base: existing loans paid on time
T["hist_critico"] = (ch == "critical/other existing credit").astype(int)
T["hist_atrasos"] = (ch == "delayed previously").astype(int)
T["hist_sin_creditos"] = ch.isin(["all paid", "no credits/all paid"]).astype(int)
sv = german["savings_status"]                                        # base: savings below 100 DM
T["ahorro_100_1000"] = sv.isin(["100<=X<500", "500<=X<1000"]).astype(int)
T["ahorro_sobre_1000"] = (sv == ">=1000").astype(int)
T["ahorro_desconocido"] = (sv == "no known savings").astype(int)
hs = german["housing"]                                               # base: owns their home
T["vivienda_arriendo"] = (hs == "rent").astype(int)
T["vivienda_gratis"] = (hs == "for free").astype(int)
pu = german["purpose"]                                               # base: other purposes
T["proposito_auto"] = pu.isin(["new car", "used car"]).astype(int)
T["proposito_radio_tv"] = (pu == "radio/tv").astype(int)
T["proposito_muebles"] = (pu == "furniture/equipment").astype(int)

XCOLS = [c for c in T.columns if c not in ("id", "y")]
NUM = ["duracion", "monto", "edad", "cuota_pct_ingreso"]
print(len(XCOLS), "explanatory columns:", XCOLS)
T.head()


### Step 2: split training and testing, and scale using training only

The 1,000 rows are split **before** doing anything else: 700 for training and 300 for
measuring (testing), at random but keeping the proportion of bad payers in both groups (this is called a
**stratified** split). The fixed seed makes it come out as the same split as in the Excel file.

Then we scale the four numeric columns: from each one we subtract its mean and divide by its
standard deviation, just as in the previous notebooks. The 0/1 columns are not scaled, so that their
coefficient reads as the effect of belonging to that category. The detail that matters: **the mean and the
standard deviation are computed only with the 700 training rows**, and those numbers are used to scale the 300
testing rows. If we used all 1,000, testing would be influencing the model, even if minimally. It is the
first hygiene rule of the course: testing takes no part in anything that counts as learning.


In [ ]:

rng = np.random.default_rng(2026)
conjunto = np.array(["training"] * len(T), dtype=object)
for clase in (0, 1):
    idx = np.where(T["y"].values == clase)[0]
    rng.shuffle(idx)
    conjunto[idx[: int(round(0.3 * len(idx)))]] = "testing"
T["conjunto"] = conjunto
tr = T["conjunto"] == "training"
te = ~tr

media = T.loc[tr, NUM].mean()
desv = T.loc[tr, NUM].std(ddof=0)      # population standard deviation, the same one StandardScaler uses
S = T.copy()
S[NUM] = (T[NUM] - media) / desv

X_tr, y_tr = S.loc[tr, XCOLS].values, S.loc[tr, "y"].values
X_te, y_te = S.loc[te, XCOLS].values, S.loc[te, "y"].values
print("training:", tr.sum(), "customers,", y_tr.sum(), "bad payers  |  testing:", te.sum(), "customers,", y_te.sum(), "bad payers")
pd.DataFrame({"training mean": media.round(3), "training std": desv.round(3)})


## 2. From the straight line to the logistic

### The naive attempt

The first idea is to use the usual linear regression with $y = 1$ if the customer did not pay and $y = 0$ if they did:

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \dots + \theta_{18} x_{18}$$

where $x_1, \dots, x_{18}$ are the prepared columns and $\theta_0, \dots, \theta_{18}$ the weights the
model fits. It half works. A straight line has no ceiling: for some customers it predicts a negative number and,
for an extreme customer, a number greater than 1; neither means anything as a probability. Let's see it.


In [ ]:

lineal = LinearRegression().fit(X_tr, y_tr)
y_lin = lineal.predict(S[XCOLS].values)
print("Minimum linear prediction:", round(y_lin.min(), 3), "  maximum:", round(y_lin.max(), 3))
print("Customers with a prediction outside [0, 1]:", int(((y_lin < 0) | (y_lin > 1)).sum()), "out of", len(y_lin))

# an extreme hypothetical customer: 72-month loan for 18,000 DM, 21 years old, high installment, negative account, past delays, renting
hip = pd.Series(0.0, index=XCOLS)
hip["duracion"], hip["monto"], hip["edad"], hip["cuota_pct_ingreso"] = 72, 18000, 21, 4
hip["cta_negativa"] = hip["hist_atrasos"] = hip["vivienda_arriendo"] = 1
hip[NUM] = (hip[NUM] - media) / desv
print("Linear prediction for the hypothetical customer:", round(float(lineal.predict(hip.values.reshape(1, -1))[0]), 3))


### The logistic function

The solution is more than 70 years old and is called **logistic regression** (Berkson, 1944). The recipe: build the
same linear combination, but instead of reading it directly as the prediction, pass it through a function that
squashes it into the range between 0 and 1.

$$z = \theta_0 + \theta_1 x_1 + \dots + \theta_{18} x_{18} \qquad\qquad p = \frac{1}{1 + e^{-z}}$$

What each piece means: $z$ is the customer's **score**, an arbitrary number, with no bounds, that summarizes
their risk; $p$ is the **estimated probability** of not paying. The second formula is the **logistic function**
or sigmoid: with $z = 0$ it gives $p = 0.5$; with $z = 3$ it gives $0.95$; with $z = -3$ it gives $0.05$; it never touches 0 or 1.
It is called an **activation** or squashing function: the line can output whatever it wants, the sigmoid
brings it into the range of a probability. (There is a cousin, the **probit**, which uses the cumulative normal and gives an
almost identical curve; the logistic is preferred because its derivative is simpler and because $z$ turns out to be exactly
the log of the odds, $\ln(p/(1-p))$, which makes the weights interpretable.)

With $p$ in hand the second piece appears: the **threshold**. To decide yes or no the probability has to be collapsed:
if $p$ exceeds the threshold, we predict "does not pay"; otherwise, "pays". For now the threshold is 0.5, the obvious value. Later
on we are going to question it.


In [ ]:

z = np.linspace(-6, 6, 121)
from scipy.stats import norm
fig = go.Figure()
fig.add_scatter(x=z, y=1 / (1 + np.exp(-z)), name="logistic 1/(1+e^-z)")
fig.add_scatter(x=z, y=norm.cdf(z), name="probit (cumulative normal)", line=dict(dash="dash"))
fig.update_layout(title="The logistic function (and the probit): any score becomes a probability",
                  xaxis_title="score z", yaxis_title="p", height=380)
fig.show()


## 3. How a model that predicts probabilities measures its errors

With a straight line we used the squared error. With probabilities there is something better. If a customer turned out bad
($y = 1$) and the model had given them $p = 0.9$, it was only slightly wrong; if it had given them $p = 0.1$, it was very
wrong. The loss $-\ln(p)$ captures that: it is 0.105 in the first case and 2.303 in the second, and it grows without
bound as $p$ approaches 0. For a good customer ($y = 0$) the loss is $-\ln(1 - p)$, the
mirror image. Written together, for customer $i$:

$$\text{loss}_i = -\left[ y_i \ln(p_i) + (1 - y_i) \ln(1 - p_i) \right]$$

It is called **binary cross-entropy** or *log loss*. The model's **cost function** is the sum (or the
average) of that loss over the training customers, and training means finding the weights $\theta$ that
minimize it. Why the logarithm and not $(y - p)^2$: because the logarithm punishes misplaced confidence. Saying
99% and being wrong costs 4.6; saying 60% and being wrong costs 0.9. A model that minimizes this loss learns not to be
more confident than the data justifies.


In [ ]:

p_grid = np.linspace(0.005, 0.995, 199)
fig = go.Figure()
fig.add_scatter(x=p_grid, y=-np.log(p_grid), name="if y = 1: −ln(p)")
fig.add_scatter(x=p_grid, y=-np.log(1 - p_grid), name="if y = 0: −ln(1 − p)")
fig.add_scatter(x=p_grid, y=(1 - p_grid) ** 2, name="squared error if y = 1: (1 − p)²", line=dict(dash="dot"))
fig.update_layout(title="Loss as a function of the predicted probability", xaxis_title="predicted p", yaxis_title="loss", height=380)
fig.show()

# two models on the same four customers: the one that is confidently wrong pays dearly
ejemplo = pd.DataFrame({"clase real y": [0, 0, 1, 1], "modelo A: p": [0.01, 0.80, 0.90, 0.10], "modelo B: p": [0.01, 0.05, 0.90, 0.80]})
for m in ("modelo A: p", "modelo B: p"):
    ejemplo[m.replace(": p", ": pérdida")] = -np.where(ejemplo["clase real y"] == 1, np.log(ejemplo[m]), np.log(1 - ejemplo[m]))
print(ejemplo.round(3))
print("Total loss A:", round(ejemplo["modelo A: pérdida"].sum(), 3), "  B:", round(ejemplo["modelo B: pérdida"].sum(), 3))


## 4. Training the model

In the Excel file the weights are found with Solver, minimizing the sum of losses in training. Here it is done by
`LogisticRegression` from scikit-learn with a finer numerical algorithm, but it is the same problem: since the
cross-entropy of a logistic regression has a single minimum, both should arrive at the same weights. We ask for
`penalty=None` because scikit-learn regularizes by default, and we only get to regularization in section 8.


In [ ]:

logit = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr, y_tr)
p_tr = logit.predict_proba(X_tr)[:, 1]
p_te = logit.predict_proba(X_te)[:, 1]

pesos = pd.Series(np.r_[logit.intercept_, logit.coef_[0]], index=["intercept"] + XCOLS, name="theta")
print(pesos.round(4))
print()
print("Sum of losses in training (the Solver objective):", round(log_loss(y_tr, p_tr) * len(y_tr), 2))
print("Average loss training:", round(log_loss(y_tr, p_tr), 4), "  testing:", round(log_loss(y_te, p_te), 4))
print("Minimum and maximum p in testing:", round(p_te.min(), 3), round(p_te.max(), 3), " (always inside (0, 1))")


How to read the weights: they are in units of $z$, that is, of log odds. `cta_negativa` at 1.55
says that having the checking account in the red multiplies the odds of not paying by $e^{1.55} \approx 4.7$
relative to having no account (the base category), all else equal. `duracion` at 0.33 says that a
loan one standard deviation longer (about 12 months) multiplies the odds by 1.4. The negative ones
protect: known and high savings, critical history (which in this dataset means "already has other loans and
is paying them"), age.


## 5. How we evaluate the model

### Accuracy and what it hides

The starting point is **accuracy**: what fraction of the customers was correctly classified with
the 0.5 threshold. But it hides what matters: in this dataset, a model that said "everyone pays" would get
70% right without having learned anything, and it would not catch a single bad payer.

### The confusion matrix

The matrix separates the four cases. With positive = bad payer (the event we want to detect):

- **TP** (true positive): was bad and we flagged them.
- **FP** (false positive): was good and we flagged them.
- **TN** (true negative): was good and we let them through.
- **FN** (false negative): was bad and we let them through.

### Recall and precision: effectiveness and efficiency

From the matrix come the two metrics you have to know how to read:

$$\text{recall} = \frac{TP}{TP + FN} \qquad\qquad \text{precision} = \frac{TP}{TP + FP}$$

**Recall**: of all the bad payers there were, what fraction did I catch. It is a measure of **effectiveness**.
**Precision**: of all those I flagged as bad, what fraction really was. It is a measure of
**efficiency**, because every false flag costs something (a good customer rejected, a useless collection call).
**F1** is the harmonic mean of the two, useful as a summary when you do not know which one matters more.

Each class has its own recall and precision; scikit-learn's `classification_report` shows them together
(the recall of class 0 is also called specificity). We measure them in training and in testing; the one that counts
is the testing one.


In [ ]:

UMBRAL = 0.5
def matriz(y_real, p, umbral=UMBRAL):
    pred = (p > umbral).astype(int)
    cm = confusion_matrix(y_real, pred, labels=[0, 1])
    return pd.DataFrame(cm, index=["actual 0 (pays)", "actual 1 (does not pay)"], columns=["predicted 0", "predicted 1"])

for nombre, yy, pp in [("TRAINING", y_tr, p_tr), ("TESTING", y_te, p_te)]:
    pred = (pp > UMBRAL).astype(int)
    print(f"=== {nombre}: accuracy {accuracy_score(yy, pred):.3f}   ('everyone pays' model: {1 - yy.mean():.3f})")
    print(matriz(yy, pp))
    print(classification_report(yy, pred, target_names=["class 0 (pays)", "class 1 (does not pay)"], digits=3))


### The trade-off between effectiveness and efficiency

With threshold 0.5 the model catches 57% of the bad payers in testing (recall) and, of those it flags, 65%
really are (precision). That pair has a trade-off that depends on the business, and there is no general
answer:

| Case | What a false negative costs (not flagging) | What a false positive costs (over-flagging) | What is favored |
|---|---|---|---|
| Credit risk | Approving someone who does not pay: the capital is lost | Rejecting someone who would have paid: the margin is lost | Recall (effectiveness), with asymmetric costs |
| Churn | Not contacting someone who was leaving: the customer is lost | Contacting someone who was staying: a call, a discount | Recall, but the retention budget is finite, so precision matters more than it seems |
| Fraud | Letting a fraudulent transaction through: the full loss | Blocking a legitimate one: an annoyed honest customer | Precision (efficiency) keeps the friction tolerable |

The lever for moving between the two is the threshold, and that is what the next section is about.


## 6. Why 50% and not another number: moving the threshold, the ROC curve and the AUC

The threshold is not part of the model; it is a decision. The next table recomputes the confusion matrix on
testing for every threshold between 0 and 1 and shows how recall, precision and F1 move. Lowering the threshold catches
more bad payers (recall goes up) at the cost of flagging more good ones (precision goes down); raising it does the opposite. There is
no threshold that improves both at once: that is the trade-off, now with numbers.


In [ ]:

def metricas_por_umbral(y_real, p, umbrales):
    filas = []
    for u in umbrales:
        pred = (p > u).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_real, pred, labels=[0, 1]).ravel()
        rec = tp / (tp + fn) if tp + fn else 0.0
        pre = tp / (tp + fp) if tp + fp else 0.0
        filas.append(dict(umbral=u, TP=tp, FP=fp, TN=tn, FN=fn, acierto=(tp + tn) / len(y_real), recall=rec, precision=pre,
                          F1=2 * rec * pre / (rec + pre) if rec + pre else 0.0, tasa_FP=fp / (fp + tn) if fp + tn else 0.0))
    return pd.DataFrame(filas)

umbrales = np.abs(np.round(np.arange(1.0, -0.001, -0.05), 2))
tabla_u = metricas_por_umbral(y_te, p_te, umbrales)
print(tabla_u.round(3).to_string(index=False))

fig = go.Figure()
for col in ("recall", "precision", "F1"):
    fig.add_scatter(x=tabla_u["umbral"], y=tabla_u[col], name=col, mode="lines+markers")
fig.update_layout(title="Recall, precision and F1 in testing as a function of the threshold", xaxis_title="threshold", yaxis_title="", height=380)
fig.show()


### The ROC curve is that same table, drawn

On the horizontal axis goes the **false positive rate** ($FP / (FP + TN)$: what fraction of the good ones
I flagged) and on the vertical axis the **recall** (also called the true positive rate). Each point is a
threshold; walking along the curve means moving the threshold from 1 (nobody flagged, bottom left corner) to 0
(everybody flagged, top right corner). A model that guesses at random follows the diagonal; a perfect one climbs
up the left edge.

The **area under the curve** (AUC) summarizes how far from the diagonal it is. It can be computed by hand as a sum
of trapezoids over the 21 points of the table; scikit-learn uses the 300 testing scores as thresholds, so
it gets a slightly finer area. Both are shown.

A complementary reading of the AUC that is worth knowing: it is the probability that, picking a bad
payer and a good payer at random, the model assigned more risk to the bad one. It is useful for comparing models without
fixing a threshold, but it hides the trade-off: when two ROC curves cross, a single number decides nothing, and
you have to look at the curve in the range of thresholds the business is going to use.


In [ ]:

# AUC by trapezoids over the table (thresholds go from 1 to 0, so the FP rate increases)
x_, y_ = tabla_u["tasa_FP"].values, tabla_u["recall"].values
auc_trapecios = float(np.sum((x_[1:] - x_[:-1]) * (y_[1:] + y_[:-1]) / 2))
auc_sklearn = roc_auc_score(y_te, p_te)
print("AUC by trapezoids (21 thresholds):", round(auc_trapecios, 4), "   AUC scikit-learn (300 thresholds):", round(auc_sklearn, 4))
print("AUC in training:", round(roc_auc_score(y_tr, p_tr), 4))

fpr, tpr, _ = roc_curve(y_te, p_te)
fig = go.Figure()
fig.add_scatter(x=fpr, y=tpr, name=f"model (AUC {auc_sklearn:.3f})", mode="lines")
fig.add_scatter(x=x_, y=y_, name="the 21 thresholds of the table", mode="markers", marker=dict(size=8))
fig.add_scatter(x=[0, 1], y=[0, 1], name="random (AUC 0.5)", line=dict(dash="dash"))
fig.update_layout(title="ROC curve in testing", xaxis_title="false positive rate", yaxis_title="recall (true positive rate)", height=450, width=550)
fig.show()


An honest detail: here the testing AUC (0.82) comes out higher than the training one (0.78). That is not the usual thing and
it does not mean the model generalizes better than it learned: with 300 testing customers the AUC has a
sampling error of several points, and in section 10 we are going to measure it.


## 7. If I know how much each error costs: the optimal threshold

If I know the cost of each outcome, the threshold stops being a discussion and becomes a calculation. The German Credit
dataset itself comes with a cost matrix: approving a bad payer ($FN$) costs 5, rejecting a good
payer ($FP$) costs 1, and getting it right costs nothing.

There are two ways to find the threshold, and they answer different questions.

**The formula.** For a customer with probability $p$ of not paying, flagging them costs on average
$(1 - p) \cdot c_{FP}$ (if they were actually going to pay) and not flagging them costs $p \cdot c_{FN}$ (if they were actually not going to
pay). It pays to flag them when $p \cdot c_{FN} > (1 - p) \cdot c_{FP}$, that is, when

$$p > \frac{c_{FP}}{c_{FP} + c_{FN}}$$

With 1 and 5 this gives 0.167. Notice that no data appears: the rule is the same for any model and
any sample, and it is exact under a single condition, that $p$ be a well **calibrated** probability, meaning
that of the customers to whom the model gives $p = 0.3$, 30% actually do not pay. (With benefits for
getting it right the formula is $(c_{FP} - b_{TN}) / ((c_{FP} - b_{TN}) + (c_{FN} - b_{TP}))$.)

**The table.** Walk through the thresholds, compute the total cost in testing with the confusion matrix of each
one, and keep the minimum. It does not assume calibration and works with any cost structure, even if the
cost depends on the customer. In exchange, it optimizes against a sample of 300, so it inherits its noise.


In [ ]:

c_TP, c_FP, c_TN, c_FN = 0, 1, 0, 5
umbral_formula = (c_FP - c_TN) / ((c_FP - c_TN) + (c_FN - c_TP))
tabla_u["costo"] = tabla_u["TP"] * c_TP + tabla_u["FP"] * c_FP + tabla_u["TN"] * c_TN + tabla_u["FN"] * c_FN
fila_min = tabla_u.loc[tabla_u["costo"].idxmin()]
print("Optimal threshold by formula:", round(umbral_formula, 3))
print("Threshold with the lowest cost in the table:", fila_min["umbral"], " cost:", int(fila_min["costo"]), "  (cost with threshold 0.5:", int(tabla_u.loc[tabla_u.umbral == 0.5, "costo"].iloc[0]), ", 'everyone pays':", int(tabla_u.loc[tabla_u.umbral == 1.0, "costo"].iloc[0]), ")")
fig = px.line(tabla_u, x="umbral", y="costo", markers=True, title="Total cost in testing as a function of the threshold (FN = 5, FP = 1)")
fig.add_vline(x=umbral_formula, line_dash="dash", annotation_text="formula 0.167")
fig.update_layout(height=360)
fig.show()


The table says 0.25 and the formula 0.167. The two possible causes are independent and leave different traces.

**Poor calibration** is a problem of the model: if the customers with $p = 0.2$ actually default
30% of the time, the formula points to the wrong place and the table corrects it. It is diagnosed by grouping testing
by deciles of $p$ and comparing the average $p$ of each group with the actual fraction of bad payers.

**Sampling variation** is a problem of the data: with 300 customers, moving the threshold from 0.25 to 0.30 changes the
classification of about 15 of them, and the cost of those 15 depends on who landed in the sample. The valley around the
minimum is flat (136, 138, 141) and with another sample the optimum jumps. It is diagnosed by resampling, which is what
we do in section 10.

Let's look at calibration.


In [ ]:

cal = pd.DataFrame({"p": p_te, "y": y_te})
cal["grupo"] = pd.qcut(cal["p"], 5, labels=["20% least risky", "2", "3", "4", "20% most risky"])
calib = cal.groupby("grupo", observed=True).agg(clientes=("y", "size"), p_promedio=("p", "mean"), fraccion_real_malos=("y", "mean"))
print(calib.round(3))
print("\nIf the model is calibrated, the last two columns look alike. With 60 customers per group, differences of 0.05 are noise.")


## 8. Regularization: charging the model for the size of its weights

The idea comes from the linear regression classes: when a model is free to move many weights, it
tends to fit the noise in training, and one way to rein it in is to add to the cost function a term
that grows with the size of the weights. With the logistic it is just as easy:

$$J(\theta) = \sum_i \text{loss}_i + \text{penalty}(\theta)$$

- **Ridge (L2)**: penalty $= \frac{\lambda}{2} \sum_j \theta_j^2$. Shrinks all weights toward zero, the large ones more, but none reaches zero.
- **Lasso (L1)**: penalty $= \lambda \sum_j |\theta_j|$. Also shrinks, but leaves weights exactly at zero: it eliminates variables.
- **Elastic net**: a mix, $\lambda \left[ \alpha \sum |\theta_j| + \frac{1 - \alpha}{2} \sum \theta_j^2 \right]$, with $\alpha$ the L1 fraction.

In all of them, $\lambda$ says how much size costs, and the intercept is left out (it says nothing about any
variable). In scikit-learn the parameter is $C = 1/\lambda$. We are going to sweep $\lambda$ and see what happens to the
weights and to performance in testing.


In [ ]:

def ajustar(pen, lam, alpha=0.5):
    if lam == 0:
        m = LogisticRegression(penalty=None, max_iter=10000)
    elif pen == "l2":
        m = LogisticRegression(penalty="l2", C=1 / lam, max_iter=10000)
    elif pen == "l1":
        m = LogisticRegression(penalty="l1", C=1 / lam, solver="saga", max_iter=50000, tol=1e-8)
    else:
        m = LogisticRegression(penalty="elasticnet", C=1 / lam, l1_ratio=alpha, solver="saga", max_iter=50000, tol=1e-8)
    m.fit(X_tr, y_tr)
    pt, pp = m.predict_proba(X_tr)[:, 1], m.predict_proba(X_te)[:, 1]
    return m, dict(lam=lam, perdida_train=log_loss(y_tr, pt), perdida_test=log_loss(y_te, pp), auc_train=roc_auc_score(y_tr, pt),
                   auc_test=roc_auc_score(y_te, pp), pesos_en_cero=int((np.abs(m.coef_[0]) < 1e-6).sum()), suma_abs_pesos=np.abs(m.coef_[0]).sum())

LAMS = [0, 1, 3, 10, 30, 100, 300]
resultados, trayectorias = {}, {}
for pen in ("l2", "l1", "en"):
    filas, tray = [], []
    for lam in LAMS:
        m, r = ajustar(pen, lam); filas.append(r); tray.append(m.coef_[0])
    resultados[pen] = pd.DataFrame(filas).set_index("lam")
    trayectorias[pen] = pd.DataFrame(np.array(tray), index=LAMS, columns=XCOLS)

for pen, nombre in [("l2", "Ridge (L2)"), ("l1", "Lasso (L1)"), ("en", "Elastic net (alpha 0.5)")]:
    print(f"=== {nombre}")
    print(resultados[pen].round(4).to_string())
    print()


In [ ]:

fig = go.Figure()
for col in XCOLS:
    fig.add_scatter(x=[str(l) for l in LAMS], y=trayectorias["l1"][col], name=col, mode="lines+markers")
fig.update_layout(title="Lasso: each weight as a function of lambda (the variables switch off one at a time)", xaxis_title="lambda", yaxis_title="theta", height=480)
fig.show()


An honest reading, worth stating as is: on this dataset **regularizing does not improve prediction**. The
testing loss goes up with $\lambda$ instead of down, and the testing AUC stays at 0.82 up to
$\lambda = 10$ and then falls. With 700 customers and 18 variables, the unregularized logistic was not
overfitting; there was nothing to rein in. What regularization does deliver here is a smaller model
(lasso with $\lambda = 10$ switches off 7 of the 18 variables and loses one hundredth of AUC) and weights that are more stable under
changes of sample. Regularization does improve testing when there are many variables relative to the rows, or
highly correlated variables, or high-degree polynomials; that is covered in the advanced material.


## 9. Imbalanced classes: what it causes and how it is corrected

In German Credit 30% of the customers are bad payers. Cross-entropy adds up each customer's error
equally, so the model cares more about getting the 490 good ones in training right than the 210 bad ones: that is
why the recall of class 0 is 0.87 and that of class 1, the one that matters, 0.57. In more
imbalanced datasets (fraud with 1% of cases) the effect is worse: the model learns to always say "no" and gets
99% right.

The solutions change one of two things: how much each customer weighs in the cost function, or which customers
go into training.

- **Class weights**: each row's loss is multiplied by a weight that depends on its class, typically
  $n / (2 \cdot n_{class})$ so that the two classes add up to the same (`class_weight="balanced"`).
- **Undersampling**: good payers are randomly thrown away until they match the bad ones, and training uses fewer rows.
- **Oversampling**: the rows of the minority class are repeated until the two classes weigh the same. In the general
  case the count is not an integer: with 490 good and 210 bad payers each bad payer would have to be repeated
  490/210 = 2.33 times. Two equivalent ways to handle it: sample 490 rows with replacement from the 210 bad payers
  (some come out twice, others three times; that is what `RandomOverSampler` in `imbalanced-learn` does), or give
  each bad payer a weight of 2.33 in the cost function (`sample_weight`), which is exactly class weights. Repeating
  a row k times always equals a weight of k, integer or not. Below are both versions: the simple one, with each bad
  payer twice, and the generic one.
- **SMOTE** (Chawla et al., 2002): it does not repeat, it invents. It takes a bad payer, finds its nearest neighbors
  among the bad payers, and creates a synthetic customer at a random point on the segment joining them:
  $x_{new} = x_a + u \cdot (x_b - x_a)$ with $u$ between 0 and 1. It is in the `imbalanced-learn` library; here
  we write it by hand in ten lines so you can see what it does.

Everything is done **on training only**: testing keeps its 300 real customers.


In [ ]:

def evaluar(m, nombre, umbral=0.5):
    p = m.predict_proba(X_te)[:, 1]
    pred = (p > umbral).astype(int)
    return dict(modelo=nombre, umbral=umbral, acierto=accuracy_score(y_te, pred), recall_1=recall_score(y_te, pred), precision_1=precision_score(y_te, pred),
                F1_1=f1_score(y_te, pred), recall_0=recall_score(y_te, pred, pos_label=0), AUC=roc_auc_score(y_te, p), p_promedio=p.mean())

n1, n0 = int(y_tr.sum()), int((y_tr == 0).sum())
comparacion = [evaluar(logit, "unbalanced")]

# 1) class weights
cw = LogisticRegression(penalty=None, max_iter=5000, class_weight="balanced").fit(X_tr, y_tr)
print("'balanced' weights: class 1 =", round(len(y_tr) / (2 * n1), 3), " class 0 =", round(len(y_tr) / (2 * n0), 3))
comparacion.append(evaluar(cw, "class weights balanced"))

# 2) undersampling: all the bad ones + n1 good ones at random
rng_b = np.random.default_rng(11)
idx_buenos = rng_b.choice(np.where(y_tr == 0)[0], size=n1, replace=False)
idx_sub = np.r_[np.where(y_tr == 1)[0], idx_buenos]
sub = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr[idx_sub], y_tr[idx_sub])
comparacion.append(evaluar(sub, f"undersampling {n1} + {n1}"))

# 3) simple oversampling: each bad payer twice (equivalent to sample_weight 2; 420 bad against 490 good)
over = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr, y_tr, sample_weight=np.where(y_tr == 1, 2.0, 1.0))
comparacion.append(evaluar(over, "oversampling, bad x2"))
# 3b) generic oversampling: sample n0 bad payers with replacement (490 against 490), like RandomOverSampler
rng_o = np.random.default_rng(7)
idx_over = np.r_[np.where(y_tr == 0)[0], rng_o.choice(np.where(y_tr == 1)[0], size=n0, replace=True)]
over_g = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr[idx_over], y_tr[idx_over])
comparacion.append(evaluar(over_g, f"random oversampling {n0} + {n0}"))

# 4) SMOTE by hand: synthetic rows until 490 and 490
X_mal = X_tr[y_tr == 1]
vecinos = NearestNeighbors(n_neighbors=6).fit(X_mal)
rng_s = np.random.default_rng(5)
sinteticos = []
for _ in range(n0 - n1):
    i = rng_s.integers(len(X_mal))
    j = rng_s.choice(vecinos.kneighbors(X_mal[i:i + 1], return_distance=False)[0][1:])
    u = rng_s.uniform()
    sinteticos.append(X_mal[i] + u * (X_mal[j] - X_mal[i]))
X_sm = np.vstack([X_tr, np.array(sinteticos)]); y_sm = np.r_[y_tr, np.ones(n0 - n1, int)]
smote = LogisticRegression(penalty=None, max_iter=5000).fit(X_sm, y_sm)
comparacion.append(evaluar(smote, f"SMOTE, {n0} + {n0}"))

# and the unbalanced model with another threshold
comparacion.append(evaluar(logit, "unbalanced", umbral=0.30))
comparacion.append(evaluar(logit, "unbalanced", umbral=0.25))
print(pd.DataFrame(comparacion).round(3).to_string(index=False))


The five methods raise the recall of class 1 from 0.57 to between 0.82 and 0.87 and lower precision from 0.65 to
between 0.50 and 0.56: none of them creates information, all of them move the operating point toward catching more bad
payers. The two versions of oversampling tell the same story: the simple one (bad x2, 420 against 490) stays a little
closer to the unbalanced model, and the generic one (490 against 490, with replacement) lands next to class weights and
SMOTE, which is what you would expect since all three equalize the weight of the two classes exactly. The AUC barely changes, because the AUC depends on neither the threshold nor the balancing.

And the row that is worth the conversation: the unbalanced model with threshold 0.30 has recall 0.856 and precision
0.513, almost the same as class weights (0.856 and 0.527). For a logistic regression, **balancing and moving the threshold
are two roads to the same place**. The difference is that balancing shifts the probabilities (the average $p$
in testing goes from 0.33 to 0.48 when the real rate is 0.30: the model stops being calibrated) and the
threshold does not. When balancing does make sense: when the minority class is so scarce that the model does not even learn to
tell it apart and the AUC itself improves with weighting, or when the algorithm does not output a probability that can be
thresholded.


## 10. How much to trust a metric: sampling variation, cross-validation and intervals

Everything above was measured on **one** testing sample of 300 customers. If the bank had drawn another
300, would the AUC still be 0.82 and the recall 0.57? The most direct way to answer is to do it: repeat the
training/testing split many times with different seeds, retrain and measure each time. That is called
**sampling variation**, and it is the reason everything that follows exists.


In [ ]:

X_all, y_all = S[XCOLS].values, S["y"].values
def entrenar_medir(semilla):
    Xa, Xb, ya, yb = train_test_split(X_all, y_all, test_size=0.3, stratify=y_all, random_state=semilla)
    m = LogisticRegression(penalty=None, max_iter=5000).fit(Xa, ya)
    p = m.predict_proba(Xb)[:, 1]
    return dict(semilla=semilla, AUC=roc_auc_score(yb, p), recall=recall_score(yb, (p > 0.5).astype(int)), precision=precision_score(yb, (p > 0.5).astype(int)))

reps = pd.DataFrame([entrenar_medir(s) for s in range(30)])
print(reps.describe().loc[["mean", "std", "min", "max"]].round(3))
fig = px.histogram(reps, x="AUC", nbins=15, title="30 different splits, 30 different AUCs (ours was 0.822)")
fig.add_vline(x=auc_sklearn, line_dash="dash"); fig.update_layout(height=340); fig.show()


The AUC moves between 0.72 and 0.81 depending on which customers landed in testing, with average 0.77, and the recall between
0.36 and 0.56. Our 0.82 and our recall of 0.57 came out above all 30 splits: the testing
sample we were dealt was a kind one, and none of the previous conclusions should rest on the AUC "being"
0.82. Two conclusions. A metric on a single split
is a **noisy estimate**, and it has to be reported with its uncertainty. And comparing two models by a
one-point difference in AUC on a single split says nothing: the difference is smaller than the noise.

### Cross-validation

Repeating random splits wastes data (each customer lands in testing many times and other times never).
$k$-fold **cross-validation** tidies up the idea: divide training into $k$ equal parts, train on
$k - 1$ and measure on the one left out, rotate $k$ times, and average. Each customer is used for measuring exactly
once. With `RepeatedStratifiedKFold` the process is repeated with several shuffles, and the standard deviation
across folds is the measure of uncertainty.

Besides estimating performance, cross-validation is the honest way to **choose** things: the threshold, the
$\lambda$ of the regularization, the number of variables. They are chosen looking only at training (from the inside, with the
folds), and testing stays untouched for the final verdict. Choosing the threshold by looking at testing, as we did in
section 7 to explain the idea, is a mild form of overfitting.


In [ ]:

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
aucs, costos_por_umbral = [], []
for i_tr, i_va in cv.split(X_tr, y_tr):
    m = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr[i_tr], y_tr[i_tr])
    p_va = m.predict_proba(X_tr[i_va])[:, 1]
    aucs.append(roc_auc_score(y_tr[i_va], p_va))
    t = metricas_por_umbral(y_tr[i_va], p_va, umbrales)
    costos_por_umbral.append((t["FP"] * c_FP + t["FN"] * c_FN).values)
aucs = np.array(aucs); costos_por_umbral = np.array(costos_por_umbral)
print(f"AUC by cross-validation (5 folds x 10 repeats, training only): {aucs.mean():.3f} +/- {aucs.std():.3f}   (in our testing it was {auc_sklearn:.3f})")
costo_medio = pd.Series(costos_por_umbral.mean(axis=0), index=umbrales)
print("Threshold with the lowest average cost in cross-validation:", costo_medio.idxmin(), " (formula: 0.167; table on testing: 0.25)")
print(costo_medio.round(1).to_string())


### Bootstrap confidence intervals

To put an error bar on the testing metric without leaving testing, we use the **bootstrap**:
resample the 300 testing customers with replacement (some come out twice, others not at all), recompute the
metric, and repeat a thousand times. The 2.5 and 97.5 percentiles of those thousand metrics are a 95% confidence
interval. It is the simplest and most general tool there is for this: it works for the AUC, the recall, the
cost or the optimal threshold, with no formulas.


In [ ]:

rng_bs = np.random.default_rng(0)
B = 1000
bs = {"AUC": [], "recall (umbral 0,5)": [], "precision (umbral 0,5)": [], "umbral óptimo (tabla)": []}
for _ in range(B):
    idx = rng_bs.integers(0, len(y_te), len(y_te))
    yb, pb = y_te[idx], p_te[idx]
    if yb.min() == yb.max():
        continue
    bs["AUC"].append(roc_auc_score(yb, pb))
    pred = (pb > 0.5).astype(int)
    bs["recall (umbral 0,5)"].append(recall_score(yb, pred)); bs["precision (umbral 0,5)"].append(precision_score(yb, pred))
    t = metricas_por_umbral(yb, pb, umbrales)
    bs["umbral óptimo (tabla)"].append(float(t.loc[(t["FP"] * c_FP + t["FN"] * c_FN).idxmin(), "umbral"]))
resumen = pd.DataFrame({k: dict(estimacion_original=v0, ic_2_5=np.percentile(v, 2.5), ic_97_5=np.percentile(v, 97.5))
                        for (k, v), v0 in zip(bs.items(), [auc_sklearn, recall_score(y_te, (p_te > .5).astype(int)), precision_score(y_te, (p_te > .5).astype(int)), 0.25])}).T
print(resumen.round(3))
fig = px.histogram(x=bs["umbral óptimo (tabla)"], nbins=20, title="Bootstrap: where the optimal threshold of the table lands if I change the testing sample")
fig.update_layout(xaxis_title="optimal threshold", height=340); fig.show()


First, a reading of the cross-validation: it estimated an AUC of 0.74, well below the 0.82 of testing. That
0.74 is the honest expectation for a new customer; the 0.82 was sampling luck, as we already saw. And the
threshold with the lowest average cost by cross-validation landed at 0.15, close to the formula (0.167) and far from the 0.25
of the table on testing.

The testing AUC is not 0.82: it is "between 0.76 and 0.87 with 95% confidence". The recall is not 0.57: it is "between
0.47 and 0.67". And the optimal threshold of the table is not 0.25: it spreads between 0.15 and 0.35 depending on which customers
land in the sample, which is exactly what section 7 anticipated. That closes the circle: the
formula (0.167) and the cross-validation threshold (0.15) fall inside the interval, and the difference with 0.25
is not evidence of anything.


## 11. What to take away

1. A straight line does not work for a yes or no because it has no ceiling; the **logistic function** brings it into $(0, 1)$ and
   turns the score into a probability. The threshold that collapses that probability into a decision is a
   separate choice.
2. A probability model is trained by minimizing **cross-entropy**, which punishes being confident and
   wrong. It has a single minimum: Solver and scikit-learn arrive at the same place.
3. Accuracy hides what matters. The **confusion matrix** and, above all, **recall** (effectiveness) and
   **precision** (efficiency) of the class of interest are what to look at, and there is a trade-off between
   them that depends on the business.
4. The lever of the trade-off is the **threshold**. The ROC curve is the threshold table drawn; the AUC summarizes it
   without fixing a threshold. With costs, the optimal threshold is computed: by formula if the model is calibrated, by
   table always, and the two answers differ because of calibration or because of sampling noise.
5. **Regularization** is added to the logistic with one more term in the cost function. On this dataset it does not
   improve prediction, and saying so is part of the method.
6. **Class imbalance** is corrected by weighting or resampling; for a logistic regression, almost all of that amounts to
   moving the threshold, and balancing decalibrates the probabilities.
7. A metric on one split is a noisy estimate. **Cross-validation** estimates it better and is
   the honest way to choose a threshold or $\lambda$; the **bootstrap** puts an interval on it. Without that, a
   one-point difference between two models means nothing.


## Exercises

**Exercise 1.** Repeat the preparation with the 20 original columns (all categoricals to dummies with
`pd.get_dummies(drop_first=True)`) and train the same unregularized logistic. Does the testing AUC improve?
Before looking at the result, write down how much you expect it to improve and why; then compare with the bootstrap
interval from section 10.

**Exercise 2.** Change the costs to $c_{FN} = 10$ and $c_{FP} = 1$ (a very conservative bank). Compute the
threshold by formula, by table on testing and by cross-validation. How many good payers does the
bank reject with that threshold, and how many bad ones does it let through?

**Exercise 3.** With lasso, sweep $\lambda$ over `[1, 3, 10, 30]` using 5-fold cross-validation
on training (not testing) to choose the best one. Report how many variables remain and the testing AUC of the
chosen model, with its bootstrap interval. Is it distinguishable from the model with all 18 variables?

**Exercise 4.** Simulate a severe imbalance: keep in training only 40 of the 210 bad payers
(at random) and the 490 good ones. Train without balancing and with `class_weight="balanced"`. Compare AUC and recall of
class 1 on the original testing set. With 8% minority, is balancing still equivalent to moving the
threshold, or does it now actually improve the AUC?


## Solutions


In [ ]:

# SOLUTION 1: the 20 columns
G = german.drop(columns=["id", "class"]).copy()
num20 = G.select_dtypes(exclude="object").columns.tolist()
X20 = pd.get_dummies(G, drop_first=True).astype(float)
X20[num20] = (X20[num20] - X20.loc[tr, num20].mean()) / X20.loc[tr, num20].std(ddof=0)
m20 = LogisticRegression(penalty=None, max_iter=10000).fit(X20[tr].values, y_tr)
p20 = m20.predict_proba(X20[te].values)[:, 1]
print("Columns:", X20.shape[1], "  Testing AUC with 20 variables:", round(roc_auc_score(y_te, p20), 3), " (with 9:", round(auc_sklearn, 3), f"; bootstrap interval of the 9: {np.percentile(bs['AUC'], 2.5):.3f} a {np.percentile(bs['AUC'], 97.5):.3f})")
print("Reading: the improvement, if any, is within the sampling noise. More columns is not more useful information by itself.")


In [ ]:

# SOLUTION 2: costs 10 and 1
cFN2, cFP2 = 10, 1
u_formula = cFP2 / (cFP2 + cFN2)
t2 = metricas_por_umbral(y_te, p_te, umbrales); t2["costo"] = t2["FP"] * cFP2 + t2["FN"] * cFN2
u_tabla = t2.loc[t2["costo"].idxmin(), "umbral"]
costos2 = []
for i_tr, i_va in cv.split(X_tr, y_tr):
    m = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr[i_tr], y_tr[i_tr])
    t = metricas_por_umbral(y_tr[i_va], m.predict_proba(X_tr[i_va])[:, 1], umbrales)
    costos2.append((t["FP"] * cFP2 + t["FN"] * cFN2).values)
u_cv = pd.Series(np.mean(costos2, axis=0), index=umbrales).idxmin()
print("Threshold by formula:", round(u_formula, 3), " by table on testing:", u_tabla, " by cross-validation:", u_cv)
fila = t2.loc[t2.umbral == u_cv].iloc[0]
print(f"With the cross-validation threshold ({u_cv}) in testing: it rejects {int(fila.FP)} good payers out of 210 and lets through {int(fila.FN)} bad ones out of 90.")


In [ ]:

# SOLUTION 3: lasso with lambda chosen by cross-validation
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
res3 = {}
for lam in [1, 3, 10, 30]:
    a = []
    for i_tr, i_va in cv5.split(X_tr, y_tr):
        m = LogisticRegression(penalty="l1", C=1 / lam, solver="saga", max_iter=50000, tol=1e-8).fit(X_tr[i_tr], y_tr[i_tr])
        a.append(roc_auc_score(y_tr[i_va], m.predict_proba(X_tr[i_va])[:, 1]))
    res3[lam] = np.mean(a)
mejor = max(res3, key=res3.get)
m3 = LogisticRegression(penalty="l1", C=1 / mejor, solver="saga", max_iter=50000, tol=1e-8).fit(X_tr, y_tr)
p3 = m3.predict_proba(X_te)[:, 1]
bs3 = [roc_auc_score(y_te[i], p3[i]) for i in (rng_bs.integers(0, len(y_te), len(y_te)) for _ in range(1000)) if y_te[i].min() != y_te[i].max()]
print("AUC by cross-validation for each lambda:", {k: round(v, 3) for k, v in res3.items()}, " -> best lambda:", mejor)
print("Variables remaining:", int((np.abs(m3.coef_[0]) > 1e-6).sum()), "out of 18.  Testing AUC:", round(roc_auc_score(y_te, p3), 3),
      f" 95% bootstrap CI: {np.percentile(bs3, 2.5):.3f} a {np.percentile(bs3, 97.5):.3f}")
print("Reading: the model with fewer variables is indistinguishable from the full one; the intervals overlap almost entirely.")


In [ ]:

# SOLUTION 4: severe imbalance (8% bad payers in training)
rng4 = np.random.default_rng(4)
malos = np.where(y_tr == 1)[0]; buenos = np.where(y_tr == 0)[0]
idx4 = np.r_[rng4.choice(malos, 40, replace=False), buenos]
X4, y4 = X_tr[idx4], y_tr[idx4]
print("Severe training set:", len(y4), "customers,", int(y4.sum()), "bad (", round(y4.mean() * 100, 1), "%)")
filas4 = []
for nombre, m in [("unbalanced", LogisticRegression(penalty=None, max_iter=5000)), ("class weights balanced", LogisticRegression(penalty=None, max_iter=5000, class_weight="balanced"))]:
    m.fit(X4, y4); p4 = m.predict_proba(X_te)[:, 1]
    for u in (0.5, 0.3, 0.1):
        pred = (p4 > u).astype(int)
        filas4.append(dict(modelo=nombre, umbral=u, AUC=roc_auc_score(y_te, p4), recall_1=recall_score(y_te, pred), precision_1=precision_score(y_te, pred, zero_division=0), p_promedio=p4.mean()))
print(pd.DataFrame(filas4).round(3).to_string(index=False))
print("Reading: the AUC is the same with and without balancing (the line ranks the same way); what changes is where the 0.5 threshold lands. With 8% minority, without balancing you have to lower the threshold to 0.1 to catch anyone; balancing does it for you. It is still equivalent to moving the threshold, but the balanced model is more convenient to use.")
